# Efficient Reasoning Transfer - T4/FP16 calibrated Colab launcher
This is the authoritative Qwen3-1.7B T4 prompt-v2 workflow. It reuses the completed model-specific budget calibration, freezes a new held-out dataset, and runs the deadline-focused main comparison at gamma 0, 0.4, and 0.6. The earlier 4B/L4, prompt-v1, and five-step gamma runs remain provenance-only evidence.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT = '/content/drive/MyDrive/efficient-reasoning-transfer'

In [ ]:
import os
import subprocess
from pathlib import Path

REPOSITORY = 'https://github.com/rgopalan01/efficient-reasoning-transfer.git'
project_path = Path(PROJECT)

def run_git(arguments, *, check=True):
    result = subprocess.run(
        ['git', *arguments], text=True, capture_output=True, check=False
    )
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print(result.stderr.strip())
    if check and result.returncode != 0:
        raise RuntimeError(
            f"git {' '.join(arguments)} failed with exit code {result.returncode}"
        )
    return result

def clone_repository(path):
    path.parent.mkdir(parents=True, exist_ok=True)
    run_git(['clone', '--branch', 'main', REPOSITORY, str(path)])

def update_repository(path):
    branch = run_git(['-C', str(path), 'branch', '--show-current']).stdout.strip()
    if branch == 'main':
        pull = run_git(['-C', str(path), 'pull', '--ff-only'], check=False)
        if pull.returncode == 0:
            return True
        print('Fast-forward pull was unavailable; checking whether safe recovery is possible.')
    else:
        print(f'Checkout is on {branch or "detached HEAD"}; aligning it with origin/main.')
    tracked = run_git(
        ['-C', str(path), 'status', '--porcelain', '--untracked-files=no']
    )
    if tracked.stdout.strip():
        print('Tracked local edits are present, so this checkout will not be overwritten.')
        return False
    run_git(['-C', str(path), 'fetch', 'origin', 'main'])
    run_git(['-C', str(path), 'checkout', '-B', 'main', 'origin/main'])
    print('Recovered the clean checkout on the current origin/main; ignored run artifacts were preserved.')
    return True

# A GitHub-hosted Colab notebook does not automatically clone its repository.
# Reuse and update a clean checkout; otherwise create a separate clone in Drive.
if (project_path / '.git').is_dir():
    if not update_repository(project_path):
        project_path = project_path.with_name(project_path.name + '-repo')
        PROJECT = str(project_path)
        if (project_path / '.git').is_dir():
            if not update_repository(project_path):
                raise RuntimeError(f'{project_path} also contains tracked edits; choose another PROJECT path.')
        elif not project_path.exists() or (project_path.is_dir() and not any(project_path.iterdir())):
            clone_repository(project_path)
        else:
            raise RuntimeError(f'{project_path} exists but is not an empty Git repository location.')
elif not project_path.exists() or (project_path.is_dir() and not any(project_path.iterdir())):
    clone_repository(project_path)
else:
    # Preserve a pre-existing non-repository folder instead of overwriting it.
    project_path = project_path.with_name(project_path.name + '-repo')
    PROJECT = str(project_path)
    if (project_path / '.git').is_dir():
        if not update_repository(project_path):
            raise RuntimeError(f'{project_path} contains tracked edits; choose another PROJECT path.')
    elif not project_path.exists() or (project_path.is_dir() and not any(project_path.iterdir())):
        clone_repository(project_path)
    else:
        raise RuntimeError(f'{project_path} exists but is not a Git repository; choose a different PROJECT path.')

os.chdir(project_path)
if not Path('pyproject.toml').is_file():
    raise RuntimeError(f'pyproject.toml is missing from {project_path}')
print('Using repository:', project_path)

In [ ]:
!python --version
!nvidia-smi
!df -h /content /content/drive/MyDrive
!python -m pip install --upgrade pip
!python -m pip install -e '.[train,dev]'
# Colab may preinstall an incompatible optional TorchAO. This project uses unquantized FP16 LoRA and does not need TorchAO.
!python -m pip uninstall -y torchao

## Repository checks and deterministic pilot data

In [ ]:
from pathlib import Path
!ruff check .
!pytest -m 'not gpu'
pilot_manifest = Path('data/generated/pilot/manifest.json')
if not pilot_manifest.exists():
    !python -m effreason.data build --config configs/t4_pilot.yaml
else:
    print('Pilot data already exists; auditing without overwriting it.')
!python -m effreason.data audit --manifest data/generated/pilot/manifest.json

## T4 hardware and one-response inference preflight
The model is always loaded in FP16, even if Colab later provides a better GPU. This cell fails before the expensive probe if CUDA or approximately 15 GB of VRAM is unavailable. The one-response run checks model loading, generation, and strict `<answer>...</answer>` compliance; it is not calibration evidence.

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('A CUDA GPU is required. Select a GPU runtime in Colab.')
properties = torch.cuda.get_device_properties(0)
print('GPU:', properties.name)
print('VRAM GiB:', round(properties.total_memory / 2**30, 2))
print('Configured precision: FP16')
if properties.total_memory < 14 * 2**30:
    raise RuntimeError('At least approximately 14 GiB VRAM is required for this T4 protocol.')

In [ ]:
!python -m effreason.evaluate --config configs/t4_pilot.yaml --condition inference_preflight_prompt_v2 --split dev --samples-per-example 1 --max-records 1 --completion-budget 2048 --output runs/qwen3_1_7b_t4_prompt_v2/inference_preflight

In [ ]:
import json
with open('runs/qwen3_1_7b_t4_prompt_v2/inference_preflight/manifest.json', encoding='utf-8') as stream:
    inference_preflight = json.load(stream)
display(inference_preflight)
if inference_preflight['status'] != 'completed':
    raise RuntimeError('Inference preflight did not complete. Do not start Stage 1.')
with open('runs/qwen3_1_7b_t4_prompt_v2/inference_preflight/predictions.jsonl', encoding='utf-8') as stream:
    preflight_rows = [json.loads(line) for line in stream if line.strip()]
if len(preflight_rows) != 1 or not preflight_rows[0]['parse_valid']:
    raise RuntimeError(f"Prompt-format preflight failed: {preflight_rows[0].get('parse_error') if preflight_rows else 'missing prediction'}")
print('T4 inference and strict answer-format preflight passed.')

## Stage 1 - completion-budget calibration
Collect four samples for each development item (72 responses with the pilot data) using Qwen3-1.7B in FP16 and the validated strict-format prompt. The initial 4,096-token cap is a probe, not the final choice. Neither the previous 4B budget nor the partial prompt-v1 responses are reused. If truncation exceeds 2%, the decision cell reuses every non-truncated prompt-v2 response and continues only truncated responses from their saved token prefixes at the next budget.

In [ ]:
!python -m effreason.evaluate --config configs/t4_pilot.yaml --condition initial_budget_probe_prompt_v2 --split dev --samples-per-example 4 --output runs/qwen3_1_7b_t4_prompt_v2/initial_budget_probe

In [ ]:
!mkdir -p analysis/qwen3_1_7b_t4_prompt_v2/calibration
!python -m effreason.calibrate budget --run-dir runs/qwen3_1_7b_t4_prompt_v2/initial_budget_probe --output analysis/qwen3_1_7b_t4_prompt_v2/calibration/budget_decision.json --candidates 512 1024 2048 4096 8192 --quantile 0.95 --headroom 1.10 --max-truncation-rate 0.02 --min-samples 50 --min-correct-samples 20

In [ ]:
import json
import shutil
import subprocess
import sys
from pathlib import Path

decision_path = Path('analysis/qwen3_1_7b_t4_prompt_v2/calibration/budget_decision.json')
with decision_path.open(encoding='utf-8') as stream:
    budget_decision = json.load(stream)
display(budget_decision)

if budget_decision['status'] == 'rerun_with_larger_probe':
    source_budget = int(budget_decision['current_probe_budget'])
    next_budget = budget_decision.get('next_probe_budget')
    if next_budget is None:
        raise RuntimeError('No larger supported probe budget remains.')
    next_budget = int(next_budget)
    archived_decision = decision_path.with_name(f'budget_decision_{source_budget}.json')
    if not archived_decision.exists():
        shutil.copy2(decision_path, archived_decision)
    extension_run = Path(f'runs/qwen3_1_7b_t4_prompt_v2/initial_budget_extension_{next_budget}')
    print(f'Continuing only truncated responses from {source_budget} to {next_budget} tokens.')
    subprocess.run(
        [
            sys.executable, '-m', 'effreason.evaluate',
            '--config', 'configs/t4_pilot.yaml',
            '--condition', f'initial_budget_extension_{next_budget}',
            '--split', 'dev',
            '--samples-per-example', '4',
            '--completion-budget', str(next_budget),
            '--extend-truncated-from', 'runs/qwen3_1_7b_t4_prompt_v2/initial_budget_probe',
            '--output', str(extension_run),
        ],
        check=True,
    )
    remaining_candidates = [
        str(candidate)
        for candidate in budget_decision['candidates']
        if int(candidate) >= next_budget
    ]
    subprocess.run(
        [
            sys.executable, '-m', 'effreason.calibrate', 'budget',
            '--run-dir', str(extension_run),
            '--output', str(decision_path),
            '--candidates', *remaining_candidates,
            '--quantile', '0.95',
            '--headroom', '1.10',
            '--max-truncation-rate', '0.02',
            '--min-samples', '50',
            '--min-correct-samples', '20',
        ],
        check=True,
    )
    with decision_path.open(encoding='utf-8') as stream:
        budget_decision = json.load(stream)
    display(budget_decision)

if budget_decision['status'] != 'selected':
    raise RuntimeError('Budget not selected. Inspect the saved decision before continuing.')
SELECTED_B = int(budget_decision['recommended_budget'])
print('Selected completion budget:', SELECTED_B)

## Budget evidence gate before training
If the selected budget equals the completed probe cap and probe truncation already satisfies the 2% rule, reuse those 72 responses instead of generating 36 redundant samples. A fresh validation run is launched only when the selected cap differs from the probe cap.

In [ ]:
import subprocess
import sys
from pathlib import Path

probe_budget = int(budget_decision['current_probe_budget'])
probe_truncation = float(budget_decision['truncation_rate'])
maximum_truncation = float(budget_decision['max_truncation_rate'])
if SELECTED_B == probe_budget and probe_truncation <= maximum_truncation:
    VALIDATION_RUN = Path(budget_decision['source_run'])
    print(
        f'Reusing the completed {budget_decision["sample_count"]}-response probe: '
        f'B={SELECTED_B} is unchanged and truncation={probe_truncation:.3%}.'
    )
else:
    VALIDATION_RUN = Path('runs/qwen3_1_7b_t4_prompt_v2/initial_calibrated_dev')
    print(f'Selected B={SELECTED_B} differs from probe B={probe_budget}; running fresh validation.')
    subprocess.run(
        [
            sys.executable, '-m', 'effreason.evaluate',
            '--config', 'configs/t4_pilot.yaml',
            '--condition', 'initial_calibrated_prompt_v2',
            '--split', 'dev',
            '--samples-per-example', '2',
            '--completion-budget', str(SELECTED_B),
            '--output', str(VALIDATION_RUN),
        ],
        check=True,
    )
print('Budget evidence run:', VALIDATION_RUN)

In [ ]:
import json
import subprocess
import sys

with (VALIDATION_RUN / 'predictions.jsonl').open(encoding='utf-8') as stream:
    validation_rows = [json.loads(line) for line in stream if line.strip()]
validation_truncation_rate = sum(bool(row['truncated']) for row in validation_rows) / len(validation_rows)
print('Budget evidence truncation rate:', validation_truncation_rate)
if validation_truncation_rate > 0.02:
    raise RuntimeError('Selected budget truncates more than 2% of the evidence responses; use the next larger candidate.')
subprocess.run(
    [
        sys.executable, '-m', 'effreason.analyze',
        '--run-dirs', str(VALIDATION_RUN),
        '--output', 'analysis/qwen3_1_7b_t4_prompt_v2/calibration/baseline',
    ],
    check=True,
)

## T4 training preflight or reuse
Reuse a completed prompt-v2 training manifest when available. Otherwise run one Qwen3-1.7B FP16 efficiency update at gamma 0.4 with microbatch 1 and four accumulation steps. This is only a hardware/trainer check and is excluded from the main results.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

preflight_candidates = [
    Path('runs/qwen3_1_7b_t4_prompt_v2/accuracy_seed11_pilot/manifest.json'),
    Path('runs/qwen3_1_7b_t4_prompt_v2/efficiency_g04_seed11_pilot/manifest.json'),
    Path('runs/qwen3_1_7b_t4_prompt_v2/training_preflight/manifest.json'),
    Path('runs/qwen3_1_7b_t4_prompt_v2/training_preflight_g02/manifest.json'),
]
completed_preflight = next(
    (path for path in preflight_candidates if path.is_file() and json.loads(path.read_text())['status'] == 'completed'),
    None,
)
if completed_preflight is None:
    subprocess.run(
        [
            sys.executable, '-m', 'effreason.train', '--config', 'configs/t4_efficiency.yaml',
            '--phase', 'pilot', '--gamma', '0.4', '--completion-budget', str(SELECTED_B),
            '--seed', '11', '--max-steps', '1', '--output',
            'runs/qwen3_1_7b_t4_prompt_v2/training_preflight_g04', '--resume', 'auto',
        ],
        check=True,
    )
    completed_preflight = Path('runs/qwen3_1_7b_t4_prompt_v2/training_preflight_g04/manifest.json')
print('Training preflight evidence:', completed_preflight)

In [ ]:
with completed_preflight.open(encoding='utf-8') as stream:
    training_preflight = json.load(stream)
display(training_preflight)
if training_preflight['status'] != 'completed':
    raise RuntimeError('T4 training preflight did not complete. Do not continue to Stage 2.')
print('T4 training preflight passed.')

## Stage 2 - freeze the deadline-focused main protocol and held-out data
The main comparison is fixed before final-test access: Qwen3-1.7B FP16, seed 11, 20 GRPO steps, completion budget 4096, one accuracy-only control, and efficiency conditions at gamma 0.4 and 0.6. The dataset has 24 training families, 12 held-out paired test families, and four harder held-out puzzles. This cell prints only counts and hashes, never test contents.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

if int(SELECTED_B) != 4096:
    raise RuntimeError(f'The frozen main protocol requires B=4096, not {SELECTED_B}.')
MAIN_MANIFEST = Path('data/generated/qwen3_1_7b_t4_main_v1/manifest.json')
existing = json.loads(MAIN_MANIFEST.read_text()) if MAIN_MANIFEST.is_file() else None
if existing is None or not existing.get('frozen', False):
    command = [sys.executable, '-m', 'effreason.data', 'build', '--config', 'configs/t4_main_accuracy.yaml', '--allow-main']
    if existing is not None:
        command.append('--force')
    subprocess.run(command, check=True)
subprocess.run([sys.executable, '-m', 'effreason.data', 'audit', '--manifest', str(MAIN_MANIFEST)], check=True)
main_manifest = json.loads(MAIN_MANIFEST.read_text())
if not main_manifest.get('frozen', False):
    subprocess.run([sys.executable, '-m', 'effreason.data', 'freeze', '--manifest', str(MAIN_MANIFEST)], check=True)
    main_manifest = json.loads(MAIN_MANIFEST.read_text())
expected_families = {'train': 24, 'dev': 6, 'test': 12, 'test_harder': 4}
actual_families = {name: int(main_manifest['splits'][name]['families']) for name in expected_families}
if not main_manifest.get('frozen') or actual_families != expected_families:
    raise RuntimeError(f'Main dataset freeze mismatch: {actual_families}')
print('Frozen main dataset hash:', main_manifest['dataset_hash'])
print('Held-out protocol family counts:', actual_families)

## Stage 3 - main accuracy-only control
Train the matched gamma-zero control for 20 steps. Checkpoints are written after every step; rerunning this cell resumes automatically.

In [ ]:
!python -m effreason.train --config configs/t4_main_accuracy.yaml --phase main --allow-main --completion-budget $SELECTED_B --seed 11 --max-steps 20 --output runs/qwen3_1_7b_t4_prompt_v2/main/accuracy_seed11_steps20 --resume auto

## Stages 4 and 5 - main efficiency conditions
Train gamma 0.4 and gamma 0.6 independently from the same pinned base model. Do not initialize either run from the accuracy adapter or from another gamma.

In [ ]:
!python -m effreason.train --config configs/t4_main_efficiency.yaml --phase main --allow-main --gamma 0.4 --completion-budget $SELECTED_B --seed 11 --max-steps 20 --output runs/qwen3_1_7b_t4_prompt_v2/main/efficiency_g04_seed11_steps20 --resume auto

In [ ]:
!python -m effreason.train --config configs/t4_main_efficiency.yaml --phase main --allow-main --gamma 0.6 --completion-budget $SELECTED_B --seed 11 --max-steps 20 --output runs/qwen3_1_7b_t4_prompt_v2/main/efficiency_g06_seed11_steps20 --resume auto

## Stage 6 - final held-out paired test evaluation
Evaluate each adapter once on the same 12 frozen test families. Each family contains original, renamed-surface, and changed-premise variants. Prediction rows are saved atomically, so rerunning resumes without regenerating completed responses.

In [ ]:
!python -m effreason.evaluate --config configs/t4_main_accuracy.yaml --allow-main --adapter runs/qwen3_1_7b_t4_prompt_v2/main/accuracy_seed11_steps20/adapter --condition accuracy_main --gamma 0 --completion-budget $SELECTED_B --samples-per-example 1 --split test --output runs/qwen3_1_7b_t4_prompt_v2/main/accuracy_seed11_steps20_test
!python -m effreason.evaluate --config configs/t4_main_efficiency.yaml --allow-main --adapter runs/qwen3_1_7b_t4_prompt_v2/main/efficiency_g04_seed11_steps20/adapter --condition efficiency_g04_main --gamma 0.4 --completion-budget $SELECTED_B --samples-per-example 1 --split test --output runs/qwen3_1_7b_t4_prompt_v2/main/efficiency_g04_seed11_steps20_test
!python -m effreason.evaluate --config configs/t4_main_efficiency.yaml --allow-main --adapter runs/qwen3_1_7b_t4_prompt_v2/main/efficiency_g06_seed11_steps20/adapter --condition efficiency_g06_main --gamma 0.6 --completion-budget $SELECTED_B --samples-per-example 1 --split test --output runs/qwen3_1_7b_t4_prompt_v2/main/efficiency_g06_seed11_steps20_test

## Stage 7 - harder held-out transfer evaluation
Evaluate the same three adapters on four frozen four-person puzzles. This is a small secondary transfer measure and should be reported separately from the primary paired test.

In [ ]:
!python -m effreason.evaluate --config configs/t4_main_accuracy.yaml --allow-main --adapter runs/qwen3_1_7b_t4_prompt_v2/main/accuracy_seed11_steps20/adapter --condition accuracy_main --gamma 0 --completion-budget $SELECTED_B --samples-per-example 1 --split test_harder --output runs/qwen3_1_7b_t4_prompt_v2/main/accuracy_seed11_steps20_harder
!python -m effreason.evaluate --config configs/t4_main_efficiency.yaml --allow-main --adapter runs/qwen3_1_7b_t4_prompt_v2/main/efficiency_g04_seed11_steps20/adapter --condition efficiency_g04_main --gamma 0.4 --completion-budget $SELECTED_B --samples-per-example 1 --split test_harder --output runs/qwen3_1_7b_t4_prompt_v2/main/efficiency_g04_seed11_steps20_harder
!python -m effreason.evaluate --config configs/t4_main_efficiency.yaml --allow-main --adapter runs/qwen3_1_7b_t4_prompt_v2/main/efficiency_g06_seed11_steps20/adapter --condition efficiency_g06_main --gamma 0.6 --completion-budget $SELECTED_B --samples-per-example 1 --split test_harder --output runs/qwen3_1_7b_t4_prompt_v2/main/efficiency_g06_seed11_steps20_harder

## Stage 8 - final CPU analysis
Recompute all metrics from saved predictions, compare each efficiency condition against the accuracy control with family-clustered bootstrap intervals, and generate figures.

In [ ]:
!python -m effreason.analyze --run-dirs runs/qwen3_1_7b_t4_prompt_v2/main/accuracy_seed11_steps20_test runs/qwen3_1_7b_t4_prompt_v2/main/efficiency_g04_seed11_steps20_test runs/qwen3_1_7b_t4_prompt_v2/main/efficiency_g06_seed11_steps20_test --training-run-dirs runs/qwen3_1_7b_t4_prompt_v2/main/accuracy_seed11_steps20 runs/qwen3_1_7b_t4_prompt_v2/main/efficiency_g04_seed11_steps20 runs/qwen3_1_7b_t4_prompt_v2/main/efficiency_g06_seed11_steps20 --output analysis/qwen3_1_7b_t4_prompt_v2/main/test
!python -m effreason.analyze --run-dirs runs/qwen3_1_7b_t4_prompt_v2/main/accuracy_seed11_steps20_harder runs/qwen3_1_7b_t4_prompt_v2/main/efficiency_g04_seed11_steps20_harder runs/qwen3_1_7b_t4_prompt_v2/main/efficiency_g06_seed11_steps20_harder --output analysis/qwen3_1_7b_t4_prompt_v2/main/harder

In [ ]:
with open('analysis/qwen3_1_7b_t4_prompt_v2/main/test/summary.json', encoding='utf-8') as stream:
    main_test_summary = json.load(stream)
with open('analysis/qwen3_1_7b_t4_prompt_v2/main/harder/summary.json', encoding='utf-8') as stream:
    main_harder_summary = json.load(stream)
print('Primary paired test results')
display(main_test_summary)
print('Secondary harder-transfer results')
display(main_harder_summary)

## Stage 9 - backup all experiment artifacts

In [ ]:
BACKUP = '/content/drive/MyDrive/efficient-reasoning-transfer-t4-backup'  # distinct destination
!mkdir -p $BACKUP
!rsync -a --checksum README.md IMPLEMENTATION_PLAN.md pyproject.toml configs data experiments.md src tests notebooks runs analysis $BACKUP/

All primary artifacts are already under the mounted project directory. The backup cell creates a second copy. The main test summaries, CSV tables, plots, prediction-level records, training metrics, and adapters remain available under `analysis/qwen3_1_7b_t4_prompt_v2/main/` and `runs/qwen3_1_7b_t4_prompt_v2/main/`. Never store credentials in the notebook.